# Capítulo 2 — Álgebra Linear Essencial para Finanças

Este capítulo usa álgebra linear para representar ativos, retornos, risco e fatores. A sequência conceitual é guiada pelo Volume I de *Market Risk Analysis: Quantitative Methods in Finance*, de Carol Alexander, sem reproduzir o texto da obra.

**Objetivos:**

- operar vetores e matrizes com significado financeiro;
- resolver sistemas, interpretar determinantes e reconhecer matrizes singulares;
- calcular variância de carteira pela forma quadrática $\sigma_p^2=\mathbf{w}'\Sigma\mathbf{w}$;
- distinguir matrizes PSD e PD, covariância e correlação;
- compreender autovalores, autovetores, decomposição espectral, Cholesky e LU;
- comparar PCA manual por `numpy.linalg` com `sklearn PCA`.

O exemplo central usa cinco ativos e mostra por que risco de carteira depende das covariâncias, não apenas das volatilidades individuais.

## Como ler este capítulo

O fio condutor é o risco de uma carteira. Em cada seção, identifique: **Objetivo** da operação matricial, **Intuição geométrica**, **Formulação**, **cálculo manual**, **implementação de biblioteca**, **exemplo financeiro**, **visualização ou diagnóstico**, **interpretação**, **Armadilhas numéricas** e **Exercício**. A pergunta recorrente é: que informação econômica está sendo preservada ou perdida por cada transformação?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import lu
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from quantfinance.linear_algebra import (
    cholesky_correlated_samples,
    cholesky_factor,
    correlation_to_covariance,
    covariance_to_correlation,
    is_positive_definite,
    nearest_valid_covariance,
    pca_from_correlation,
    pca_from_covariance,
)
from quantfinance.portfolio import portfolio_variance, portfolio_volatility

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 2.1 — Vetores, matrizes, operações e sistemas lineares

Um vetor pode representar preços, retornos ou pesos. Uma matriz organiza relações entre vários ativos. Para uma matriz $A$ e vetor $x$, a multiplicação $Ax$ combina as colunas de $A$ usando os elementos de $x$.

A transposta troca linhas e colunas, $A'_{ij}=A_{ji}$. O determinante indica se a transformação preserva volume e, em particular, se uma matriz quadrada é invertível:

$$\det(A)\ne 0 \Longleftrightarrow A^{-1}\text{ existe}.$$

Uma matriz singular não possui inversa. Para resolver $Ax=b$, devemos preferir um solver numérico a calcular explicitamente $A^{-1}b$.

**Interpretação financeira:** um sistema linear aparece na estimação de exposições, hedge ratios e decomposições de fatores.

**Exercício:** crie uma matriz singular e observe que `np.linalg.solve` não consegue produzir uma solução única.

In [ ]:
asset_vector = np.array([0.10, 0.15, 0.20, 0.08, 0.12])
A = np.array([
    [2.0, 1.0, 0.0],
    [1.0, 3.0, 1.0],
    [0.0, 1.0, 2.0],
])
b = np.array([5.0, 7.0, 4.0])

print("Vetor de volatilidades:", asset_vector)
print("A @ b:", A @ b)
print("A.T @ b:", A.T @ b)
print("A.T =\n", A.T)
print("det(A) =", np.linalg.det(A))
print("A^-1 =\n", np.linalg.inv(A))
print("Solução Ax=b:", np.linalg.solve(A, b))

singular = np.array([[1.0, 2.0], [2.0, 4.0]])
print("det(matriz singular) =", np.linalg.det(singular))
try:
    np.linalg.solve(singular, [1.0, 2.0])
except np.linalg.LinAlgError as error:
    print("Sistema singular rejeitado:", error)

## 2.2 — Formas quadráticas e risco de carteira

Uma forma quadrática combina um vetor com uma matriz:

$$q(\mathbf{w})=\mathbf{w}'A\mathbf{w}.$$

Para uma carteira, a matriz relevante é a covariância $\Sigma$:

$$\sigma_p^2=\mathbf{w}'\Sigma\mathbf{w}, \qquad \sigma_p=\sqrt{\mathbf{w}'\Sigma\mathbf{w}}.$$

A expressão contém variâncias individuais e covariâncias cruzadas:

$$\sigma_p^2=\sum_i w_i^2\sigma_i^2+\sum_{i\ne j}w_iw_j\sigma_{ij}.$$

Por isso não podemos calcular risco como média ponderada das volatilidades, $\sum_i w_i\sigma_i$: essa média ignora dependência entre ativos e, em geral, não é a volatilidade de nenhuma carteira.

**Interpretação financeira:** diversificação depende de como os ativos se movem juntos. Correlações baixas ou negativas podem reduzir bastante o risco agregado.

**Exercício:** compare a carteira com correlação positiva com uma carteira cuja correlação entre dois ativos seja negativa.

In [ ]:
volatilities = np.array([0.10, 0.15, 0.20, 0.08, 0.12])
weights = np.array([0.20, 0.25, 0.20, 0.15, 0.20])
correlation = np.array([
    [1.00, 0.35, 0.10, 0.20, 0.25],
    [0.35, 1.00, 0.45, 0.15, 0.30],
    [0.10, 0.45, 1.00, 0.05, 0.20],
    [0.20, 0.15, 0.05, 1.00, 0.25],
    [0.25, 0.30, 0.20, 0.25, 1.00],
])

covariance = correlation_to_covariance(correlation, volatilities)
variance = portfolio_variance(weights, covariance)
volatility = portfolio_volatility(weights, covariance)
naive_average = weights @ volatilities

print("Pesos somam:", weights.sum())
print("Variância da carteira w' Sigma w =", variance)
print("Volatilidade da carteira =", volatility)
print("Média ponderada das volatilidades =", naive_average)
print("Diferença entre risco real e média ingênua =", volatility - naive_average)
print("Risco real inclui covariâncias:", volatility != naive_average)

## 2.3 — Covariância, correlação e a relação $\Sigma=DCD$

A covariância tem unidades de retorno ao quadrado e depende da escala dos ativos. A correlação normaliza essa escala:

$$C_{ij}=\frac{\Sigma_{ij}}{\sigma_i\sigma_j}.$$

Se $D=\operatorname{diag}(\sigma_1,\ldots,\sigma_n)$, então

$$\Sigma=DCD.$$

Uma matriz de covariância válida deve ser simétrica e positiva semidefinida. Para Cholesky, precisamos de uma matriz positiva definida. Uma matriz de correlação tem diagonal igual a 1 e entradas entre -1 e 1, mas essas condições isoladas não garantem consistência conjunta.

**Interpretação financeira:** a correlação separa a escala de risco de cada ativo da estrutura de dependência entre eles.

**Exercício:** altere uma correlação fora da diagonal e verifique se a matriz ainda é positiva definida.

In [ ]:
covariance_from_dcd = correlation_to_covariance(correlation, volatilities)
eigenvalues, eigenvectors = np.linalg.eigh(covariance_from_dcd)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]
reconstructed_covariance = eigenvectors @ np.diag(eigenvalues) @ eigenvectors.T
explained = eigenvalues / eigenvalues.sum()

print("Autovalores em ordem decrescente:", eigenvalues)
print("Autovetores (colunas):\n", eigenvectors)
print("Variância explicada:", explained)
print("Reconstrução espectral exata:", np.allclose(reconstructed_covariance, covariance_from_dcd))

## 2.4 — Autovalores, autovetores e decomposição espectral

Um autovetor $v$ de uma matriz $A$ mantém sua direção após a transformação, mudando apenas de escala:

$$Av=\lambda v.$$

O escalar $\lambda$ é o autovalor. Geometricamente, os autovetores indicam eixos principais da transformação; em uma matriz de covariância, esses eixos são fatores ortogonais de risco e os autovalores medem a variância em cada direção.

Para uma matriz simétrica,

$$\Sigma=Q\Lambda Q',$$

onde $Q$ contém autovetores ortonormais e $\Lambda$ é diagonal com autovalores.

**Interpretação financeira:** os primeiros fatores podem concentrar grande parte do risco conjunto dos ativos.

**Exercício:** reconstrua a covariância usando apenas os dois maiores autovalores e compare com a matriz original.

In [ ]:
asset_names = ["Ações", "Crédito", "Commodities", "Caixa", "Tecnologia"]
D = np.diag(volatilities)
covariance_from_dcd = D @ correlation @ D

simulated_returns = cholesky_correlated_samples(correlation, 1_000, seed=42)
simulated_returns = simulated_returns * volatilities
returns = pd.DataFrame(simulated_returns, columns=asset_names)
sample_covariance = returns.cov()
sample_correlation = returns.corr()

print("Covariância populacional construída por D C D:")
display(pd.DataFrame(covariance_from_dcd, index=asset_names, columns=asset_names))
print("\nCovariância amostral simulada:")
display(sample_covariance)
print("\nCorrelação amostral simulada:")
display(sample_correlation)
print("\nIdentidade D C D:", np.allclose(covariance_from_dcd, correlation_to_covariance(correlation, volatilities)))
print("Reconstrução pela covariância:", np.allclose(covariance_to_correlation(covariance_from_dcd), correlation))
print("Matriz positiva definida:", is_positive_definite(covariance_from_dcd))

## 2.5 — Cholesky e LU

Se $\Sigma$ é positiva definida, a decomposição de Cholesky escreve

$$\Sigma=LL',$$

com $L$ triangular inferior. Ela é útil para simular vetores correlacionados: se $z\sim N(0,I)$, então $Lz$ possui a covariância desejada.

A decomposição LU escreve uma matriz como $PA=LU$, separando permutação, parte triangular inferior e parte triangular superior. Ela é útil para resolver muitos sistemas com a mesma matriz.

**Interpretação financeira:** Cholesky transforma choques independentes em choques com dependência; LU organiza soluções lineares de forma eficiente.

**Exercício:** gere amostras correlacionadas para os cinco ativos e compare a correlação amostral com a matriz-alvo.

In [ ]:
C = cholesky_factor(covariance_from_dcd)
print("Cholesky L:\n", C)
print("L L' reconstrói Sigma:", np.allclose(C @ C.T, covariance_from_dcd))

P, L, U = lu(covariance_from_dcd)
print("\nPermutação P:\n", P)
print("L triangular inferior:\n", L)
print("U triangular superior:\n", U)
print("PA = LU:", np.allclose(P @ covariance_from_dcd, L @ U))

correlated_samples = cholesky_correlated_samples(correlation, 100_000, seed=123)
print("\nCorrelação amostral dos choques correlacionados:")
display(pd.DataFrame(correlated_samples, columns=asset_names).corr())

## 2.6 — PCA por covariância e por correlação

A PCA escolhe direções ortogonais que maximizam a variância explicada. Para dados centrados, essas direções são os autovetores da matriz de covariância. Os loadings combinam direção e escala:

$$\text{loading}_{ij}=q_{ij}\sqrt{\lambda_j}.$$

A PCA pela covariância preserva a escala original. A PCA pela correlação primeiro padroniza os ativos, dando peso comparável a ativos com volatilidades diferentes.

O procedimento será feito em duas etapas:

1. `numpy.linalg.eigh` para obter autovalores e autovetores;
2. `sklearn.decomposition.PCA` como implementação de biblioteca.

Com poucos componentes, os dados podem ser reconstruídos aproximadamente:

$$X\approx \bar X + (X-\bar X)Q_kQ_k'.$$

**Interpretação financeira:** poucos fatores podem explicar grande parte dos movimentos conjuntos, reduzindo a dimensão de modelos de risco.

**Exercício:** compare a reconstrução com um e dois componentes e avalie o erro médio quadrático.

In [ ]:
X = simulated_returns
X_centered = X - X.mean(axis=0)

manual_covariance = pca_from_covariance(np.cov(X, rowvar=False))
manual_correlation = pca_from_correlation(np.corrcoef(X, rowvar=False))

scores = X_centered @ manual_covariance.eigenvectors
reconstructed_one = scores[:, :1] @ manual_covariance.eigenvectors[:, :1].T + X.mean(axis=0)
reconstructed_two = scores[:, :2] @ manual_covariance.eigenvectors[:, :2].T + X.mean(axis=0)

sklearn_pca = PCA().fit(X)

print("PCA manual por covariância - variância explicada:", manual_covariance.explained_variance_ratio)
print("PCA manual por correlação - variância explicada:", manual_correlation.explained_variance_ratio)
print("Loadings dos dois primeiros componentes:\n", manual_covariance.loadings[:, :2])
print("PCA sklearn - variância explicada:", sklearn_pca.explained_variance_ratio_[:2])
print("Comparação manual/sklearn:", np.allclose(
    manual_covariance.explained_variance_ratio[:2],
    sklearn_pca.explained_variance_ratio_[:2],
))
print("Erro MSE com 1 componente:", np.mean((X - reconstructed_one) ** 2))
print("Erro MSE com 2 componentes:", np.mean((X - reconstructed_two) ** 2))

plt.figure(figsize=(8, 4))
plt.bar(np.arange(1, 6), manual_covariance.explained_variance_ratio, label="Covariância")
plt.bar(np.arange(1, 6), manual_correlation.explained_variance_ratio, alpha=0.5, label="Correlação")
plt.xlabel("Componente principal")
plt.ylabel("Variância explicada")
plt.legend()
plt.title("Variância explicada pela PCA")
plt.show()

## Exercícios integradores

1. Crie uma matriz singular e explique por que não existe solução única para todo sistema $Ax=b$.
2. Verifique numericamente que $\Sigma=DCD$ para os cinco ativos.
3. Compare a volatilidade $\sqrt{\mathbf{w}'\Sigma\mathbf{w}}$ com $\sum_i w_i\sigma_i$ e explique a diferença.
4. Gere amostras correlacionadas por Cholesky e compare a correlação amostral com a matriz-alvo.
5. Reconstrua a covariância espectral usando somente os dois maiores autovalores.
6. Compare PCA baseada em covariância com PCA baseada em correlação.
7. Compare a reconstrução dos dados usando um e dois componentes principais.
8. Interprete economicamente os loadings do primeiro componente.

## Leitura crítica dos resultados

Verifique dimensões, simetria e unidades antes de interpretar uma matriz. Uma covariância tem unidade de retorno ao quadrado; uma correlação não tem unidade. Autovalores negativos indicam que a matriz não pode ser uma covariância válida; loadings grandes não são automaticamente previsões. Refaça a análise com pesos diferentes e explique qual conclusão permanece.